# Where visibility under-captures clicks — capstone

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/capstone.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Title and five-sentence abstract

This study asks which visible pages merit a click-capture review after accounting for observed search position. Using the FlyRank anonymized starter snapshot, we built a position-band reference and two regression models on 16,726 eligible pages, with separate client groups for training, validation, and testing. Random Forest was selected on validation error and reduced impression-weighted CTR MAE from 0.2042 to 0.1726 percentage points on 3,913 pages from six held-out clients, a 15.5% reduction. The resulting ranked queue identifies reference click shortfalls and assigns reason-coded human review actions, while calibration and subgroup checks expose where the reference can be wrong. These are observed decision-support recommendations from one cross-sectional snapshot, with no claim of future performance, recoverable clicks, or causal refresh benefit.

In [2]:
receipt_path=ROOT/'work/outputs/ctr_metrics.json'
if not receipt_path.exists():
    study.run()
r=json.loads(receipt_path.read_text())
print('Measured result receipt loaded:',r['status'])

Measured result receipt loaded: measured


## Introduction / problem statement

Which visible pages merit a human click-capture review? Core lane: CTR / Engagement Opportunity Scoring.

In [3]:
print('Research decision: rank review candidates, not automated content changes.')

Research decision: rank review candidates, not automated content changes.


## Data

Permitted starter release at pinned source commit, one page per row and trailing 90-day snapshot with unknown absolute endpoints.

In [4]:
d,audit=study.read_data()
print(json.dumps(audit,indent=2))

{
  "raw_rows": 30000,
  "raw_clients": 32,
  "raw_columns": 44,
  "duplicate_content_rows": 0,
  "excluded_below_500": 13274,
  "excluded_position_after_volume": 0,
  "eligible_rows": 16726,
  "eligible_clients": 28,
  "position_zero_raw": 1205,
  "ctr_rounding_max_pp": 0.005000000000000782,
  "sha256": "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396",
  "source_git_blob": "2f2f41fdb07868dc5599449ce9d817f58f3b2ae9",
  "source_commit": "3b657dda40c1b668072a53371883fd4413d60f08",
  "date_window": "Trailing 90-day snapshot; absolute export/end dates not present in the starter CSV. Warehouse dates are not assigned to this slice.",
  "missing_pct_eligible": {
    "word_count": 28.393,
    "search_volume": 1.602,
    "cpc": 1.602,
    "competition": 1.602,
    "main_intent": 1.262
  }
}


## Methodology

Explicit feature allowlist, position-band baseline, two regressors, disjoint client folds, validation-only selection and training-only transforms.

In [5]:
print('Features:',study.FEATURES)
print(json.dumps(r['leakage_checks'],indent=2))

Features: ['avg_position', 'log_word_count', 'content_age_days', 'days_since_last_update', 'log_search_volume', 'competition', 'log_cpc', 'content_type', 'main_intent']
{
  "client_overlap": 0,
  "content_overlap": 0,
  "target_or_identifier_features": 0,
  "preprocessing_fit_on_train_only": true,
  "winner_selected_on_validation_only": true,
  "future_forecast_claim": false
}


## Results versus baseline

Run the measured analysis top to bottom, then inspect every method on the same test set.

In [6]:
r=study.run()
print(pd.DataFrame(r['test']).T.to_string())

{
  "selected_method": "random_forest",
  "eligible_rows": 16726,
  "splits": {
    "train": {
      "rows": 8483,
      "clients": 17
    },
    "validation": {
      "rows": 4330,
      "clients": 5
    },
    "test": {
      "rows": 3913,
      "clients": 6
    }
  },
  "test": {
    "position_baseline": {
      "mae_pp": 0.2013234607495584,
      "weighted_mae_pp": 0.2042125527753737,
      "rmse_pp": 0.2870019252129417,
      "observed_ctr_pp": 0.2850271116236826,
      "predicted_ctr_pp": 0.33993391818137225,
      "n": 3913
    },
    "random_forest": {
      "mae_pp": 0.172949732444668,
      "weighted_mae_pp": 0.17261499349796033,
      "rmse_pp": 0.2742262173661863,
      "observed_ctr_pp": 0.2850271116236826,
      "predicted_ctr_pp": 0.26412428668911,
      "n": 3913
    },
    "hist_gradient_boosting": {
      "mae_pp": 0.17107748554758137,
      "weighted_mae_pp": 0.17499545088407836,
      "rmse_pp": 0.27630515157474955,
      "observed_ctr_pp": 0.2850271116236826,
     

## Limitations and honest framing

Cross-sectional association, incomplete metadata, unknown query/SERP factors, only six test clients; no future or causal label.

In [7]:
print(json.dumps(r['bootstrap'],indent=2))
print(pd.DataFrame(r['subgroup_errors']).to_string(index=False))

{
  "difference_selected_minus_baseline_pp": -0.031597559277413347,
  "client_bootstrap_95pct_interval_pp": [
    -0.04498797036979892,
    -0.02151858476015307
  ],
  "replicates": 2000,
  "clusters": 6,
  "interpretation": "Exploratory client resampling with few clusters; negative favors selected method."
}
  group_field              group    n  selected_mae_pp  baseline_mae_pp
position_band                1–3   67         0.178325         0.323867
position_band               4–10 1477         0.198743         0.209767
position_band              11–20 1209         0.161205         0.219231
position_band              21–50 1096         0.097904         0.113951
position_band                >50   64         0.104700         0.096765
 content_type comparison article   70         0.293889         0.308041
 content_type    keyword article 3828         0.171605         0.203335


## Ranked recommendations

Positive reference shortfall times observed exposure ranks human review. Local mapping is in the gitignored queue.

In [8]:
print(pd.DataFrame(r['recommendations'])[['rank','review_id','reference_gap_clicks','action']].to_string(index=False))

 rank review_id  reference_gap_clicks                        action
    1 Review 01                1403.3    Metadata and intent review
    2 Review 02                 910.5    Metadata and intent review
    3 Review 03                 465.6    Metadata and intent review
    4 Review 04                 366.5    Metadata and intent review
    5 Review 05                 339.7    Metadata and intent review
    6 Review 06                 312.3    Metadata and intent review
    7 Review 07                 276.1    Metadata and intent review
    8 Review 08                 271.7    Metadata and intent review
    9 Review 09                 231.8 Visibility and content review
   10 Review 10                 230.7    Metadata and intent review
   11 Review 11                 216.8    Metadata and intent review
   12 Review 12                 211.6    Metadata and intent review
   13 Review 13                 191.6    Metadata and intent review
   14 Review 14                 187.7    Metadat

## Reproducibility and artifacts

Source, versions, fixed seed and fingerprint make this run inspectable. Figures are linked below.

In [9]:
print(json.dumps(r['versions'],indent=2))
print('Figures:',sorted(p.name for p in (ROOT/'work/figures').glob('*.svg')))

{
  "python": "3.12.14",
  "numpy": "2.3.5",
  "pandas": "2.2.3",
  "sklearn": "1.8.0"
}
Figures: ['action_mix.svg', 'calibration.svg', 'feature_sensitivity.svg', 'model_comparison.svg']


## Acknowledgments and data credit

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [10]:
print('Built on the FlyRank ML Internship dataset: https://flyrank.ai')

Built on the FlyRank ML Internship dataset: https://flyrank.ai


## ML-12 — Five-minute demo

0:00 question and scope; 0:45 data exclusions; 1:30 client split and leakage; 2:30 comparison and calibration; 3:30 queue and reasons; 4:30 limits and rerun path.

In [11]:
print('Demo checklist complete: question, method, baseline, measured result, recommendations, limits.')

Demo checklist complete: question, method, baseline, measured result, recommendations, limits.


## ML-12 — Social post and employer-facing summary

I built a reproducible CTR review capstone on FlyRank’s anonymized starter data. A grouped Random Forest reference lowered weighted CTR error by 15.5% on unseen clients, and produces a reason-coded review queue. The study is observational; reference gaps are not promises of uplift.

Employer summary: I designed a client-separated data contract and compared a transparent baseline with two regression models. I checked calibration, subgroup errors, leakage and uncertainty. I shipped executed notebooks, reproducible source and a public paper with a ranked review playbook.

In [12]:
print('Submission is the repository URL; the exact paper URL is recorded in submission/paper_url.txt.')

Submission is the repository URL; the exact paper URL is recorded in submission/paper_url.txt.
